# 05 - Model Baseline
**Phase 2: Establish baseline model for H&M recommendations**

### What this notebook does:
1. Load curated data from Phase 1
2. Create temporal train/test split (no data leakage!)
3. Build Popularity Baseline model
4. Evaluate with MAP@12 and Recall@12
5. Track experiment with MLflow

### Why a baseline?
Any "real" model must beat this simple baseline to be useful. If we can't beat "recommend popular items to everyone", our fancy model is worthless.

In [0]:
# ========================================
# SETUP & CONFIGURATION
# ========================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import mlflow
import mlflow.spark

# Storage Configuration (same as Phase 1)
STORAGE_ACCOUNT = "ragadziyada"
STORAGE_KEY = "qIXjwo7EGK8an84BPCk446tqY9L7K7r3a9W2WB+voe2vUCvW1SK3qc/7UGOicKyBAtrptYcVfTB7+AStvWZi0A=="

PROCESSED_CONTAINER = "processed-p1"
CURATED_CONTAINER = "curated-p1"

# Set storage key
spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    STORAGE_KEY
)

# Input paths (from Phase 1)
TRANSACTIONS_CLEAN = f"abfss://{PROCESSED_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/transactions_clean/"
CUSTOMER_FEATURES = f"abfss://{CURATED_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/customer_features/"
ARTICLE_FEATURES = f"abfss://{CURATED_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/article_features/"
RECOMMENDATION_DATA = f"abfss://{CURATED_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/recommendation_dataset/"

# Output paths (for Phase 2)
MODEL_OUTPUT = f"abfss://{CURATED_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/models/"

# Recommendation settings
K = 12  # Top-K recommendations (H&M competition standard)

print("✅ Configuration loaded successfully!")
print(f"📦 Storage account: {STORAGE_ACCOUNT}")
print(f"🎯 Recommendation K: {K}")

✅ Configuration loaded successfully!
📦 Storage account: ragadziyada
🎯 Recommendation K: 12


In [0]:
# ========================================
# REPRODUCIBILITY SETTINGS
# ========================================
# Ensures consistent results across runs

import random
import numpy as np

# Set random seeds for reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Spark doesn't use random seeds for deterministic operations,
# but we document our split strategy for reproducibility:
# - Temporal split on t_dat (date-based, deterministic)
# - No random sampling involved
# - Same split date (2020-09-15) used across all notebooks

print("✅ Reproducibility settings configured!")
print(f"   Random seed: {RANDOM_SEED}")
print(f"   Split strategy: Temporal (date-based, deterministic)")
print(f"   Split date: 2020-09-15 (fixed across all notebooks)")

✅ Reproducibility settings configured!
   Random seed: 42
   Split strategy: Temporal (date-based, deterministic)
   Split date: 2020-09-15 (fixed across all notebooks)


In [0]:
# ========================================
# LOAD DATA FROM PHASE 1
# ========================================

# Load transactions (need t_dat for temporal split)
transactions_df = spark.read.parquet(TRANSACTIONS_CLEAN)

# Load feature tables
customer_features_df = spark.read.parquet(CUSTOMER_FEATURES)
article_features_df = spark.read.parquet(ARTICLE_FEATURES)

# Get row counts
trans_count = transactions_df.count()
cust_count = customer_features_df.count()
art_count = article_features_df.count()

print("=" * 50)
print("📊 DATA LOADED FROM PHASE 1")
print("=" * 50)
print(f"Transactions:     {trans_count:,} rows")
print(f"Customer features: {cust_count:,} rows")
print(f"Article features:  {art_count:,} rows")

# Check date range in transactions
date_stats = transactions_df.select(
    F.min("t_dat").alias("min_date"),
    F.max("t_dat").alias("max_date")
).collect()[0]

print(f"\n📅 Transaction date range:")
print(f"   From: {date_stats['min_date']}")
print(f"   To:   {date_stats['max_date']}")

📊 DATA LOADED FROM PHASE 1
Transactions:     28,813,419 rows
Customer features: 1,362,281 rows
Article features:  104,547 rows

📅 Transaction date range:
   From: 2018-09-20
   To:   2020-09-22


In [0]:
# ========================================
# TEMPORAL TRAIN/TEST SPLIT
# ========================================
# Why temporal split?
# - Simulates real-world: "Given past purchases, predict future purchases"
# - Random split would leak future data into training = cheating!

from datetime import date

# Latest date in dataset
max_date = date_stats['max_date']

# Test set: last 7 days of transactions
# Train set: everything before that
split_date = date(2020, 9, 15)  # 7 days before 2020-09-22

print(f"📅 Temporal Split Strategy:")
print(f"   Train: 2018-09-20 to {split_date} (everything before)")
print(f"   Test:  {split_date} to 2020-09-22 (last 7 days)")
print()

# Create train/test splits
train_df = transactions_df.filter(F.col("t_dat") < F.lit(split_date))
test_df = transactions_df.filter(F.col("t_dat") >= F.lit(split_date))

# Count rows
train_count = train_df.count()
test_count = test_df.count()

print(f"✅ Split complete!")
print(f"   Train transactions: {train_count:,}")
print(f"   Test transactions:  {test_count:,}")
print(f"   Test ratio: {test_count/train_count*100:.2f}%")

📅 Temporal Split Strategy:
   Train: 2018-09-20 to 2020-09-15 (everything before)
   Test:  2020-09-15 to 2020-09-22 (last 7 days)

✅ Split complete!
   Train transactions: 28,571,904
   Test transactions:  241,515
   Test ratio: 0.85%


In [0]:
# ========================================
# CREATE GROUND TRUTH
# ========================================
# For each customer in test set, what articles did they ACTUALLY buy?
# This is what we're trying to predict!

test_ground_truth = (
    test_df
    .groupBy("customer_id")
    .agg(F.collect_set("article_id").alias("actual_articles"))
)

# How many customers made purchases in test period?
test_customers_count = test_ground_truth.count()

print(f"✅ Ground truth created!")
print(f"   Customers who purchased in test period: {test_customers_count:,}")

# Preview: show a few customers and what they bought
print(f"\n📋 Sample ground truth (what customers actually bought):")
display(test_ground_truth.limit(5))

✅ Ground truth created!
   Customers who purchased in test period: 75,481

📋 Sample ground truth (what customers actually bought):


customer_id,actual_articles
4d28266352d06bcab3e3c13239771b41c5e3ee77e5f8f9050f62ec7b35ad46e0,"List(0776179001, 0456163084)"
bc3f87d81f42fbb6cdb9e8269532ec1faee42257dc0f0f44ce21bd6b4c11893e,"List(0881919002, 0929508001)"
f78c1a9e4a866bc1cce242c72000339fb5f7e6ff814d9466af8bfe6dd1cd16cd,"List(0730683050, 0803757005, 0869331006, 0456163086, 0803757001, 0887771001, 0809961002, 0853881001)"
ccff3c53e511b61e4e3d1b80e82ddbd2d8370cda97c85406b6b097edfa8a3f70,"List(0868691001, 0809961003)"
f2f2d11004b7af5a324d2e8c8d848a95b28df611b7d092601e0214df46d159cc,"List(0929980006, 0573085028, 0591466033, 0903840002, 0808246004, 0841544014, 0884901001)"


In [0]:
# ========================================
# POPULARITY BASELINE MODEL
# ========================================
# Simplest possible model: recommend top-K most popular items to EVERYONE
# 
# Why this matters:
# - If we can't beat this, our "smart" model is useless
# - Surprisingly hard to beat in real-world recommendation systems!

# Calculate article popularity from TRAINING data only (no cheating!)
article_popularity = (
    train_df
    .groupBy("article_id")
    .agg(F.count("*").alias("purchase_count"))
    .orderBy(F.desc("purchase_count"))
)

# Get top-K most popular articles (using DataFrame operations, not RDD)
top_k_df = article_popularity.limit(K)

# Convert to pandas to extract the list (allowed on shared clusters)
top_k_pandas = top_k_df.toPandas()
top_k_articles = top_k_pandas["article_id"].tolist()

print(f"🏆 TOP {K} MOST POPULAR ARTICLES (Training Period)")
print("=" * 50)
for i, row in top_k_pandas.iterrows():
    print(f"   {i+1:2}. Article {row['article_id']}: {row['purchase_count']:,} purchases")

print("=" * 50)
print(f"\n✅ Baseline model ready!")
print(f"   Strategy: Recommend these {K} items to ALL customers")

🏆 TOP 12 MOST POPULAR ARTICLES (Training Period)
    1. Article 0706016001: 42,337 purchases
    2. Article 0706016002: 30,624 purchases
    3. Article 0372860001: 29,101 purchases
    4. Article 0610776002: 25,059 purchases
    5. Article 0759871002: 23,779 purchases
    6. Article 0372860002: 22,234 purchases
    7. Article 0464297007: 21,750 purchases
    8. Article 0399223001: 19,570 purchases
    9. Article 0720125001: 18,908 purchases
   10. Article 0610776001: 18,662 purchases
   11. Article 0673396002: 18,374 purchases
   12. Article 0706016003: 18,362 purchases

✅ Baseline model ready!
   Strategy: Recommend these 12 items to ALL customers


In [0]:
# ========================================
# GENERATE PREDICTIONS
# ========================================
# For baseline: give EVERY customer the same top-12 popular items

# Get all customers from test set
test_customers = test_ground_truth.select("customer_id")

# Create array of top-K articles
top_k_array = F.array([F.lit(article_id) for article_id in top_k_articles])

# Assign same recommendations to all customers
predictions_df = test_customers.withColumn("predicted_articles", top_k_array)

print(f"✅ Predictions generated!")
print(f"   Customers: {predictions_df.count():,}")
print(f"   Each customer gets the same {K} recommendations")
print(f"\n📋 Sample predictions:")
display(predictions_df.limit(5))

✅ Predictions generated!
   Customers: 75,481
   Each customer gets the same 12 recommendations

📋 Sample predictions:


customer_id,predicted_articles
a1cee93a64ebeb735ab0595ef2c2c69e8a084326a2980bc79582677349464fbc,"List(0706016001, 0706016002, 0372860001, 0610776002, 0759871002, 0372860002, 0464297007, 0399223001, 0720125001, 0610776001, 0673396002, 0706016003)"
032e069cc3127b0c9c421c36150a8a6cdb7a32c2f501da6182566a943eda6a6a,"List(0706016001, 0706016002, 0372860001, 0610776002, 0759871002, 0372860002, 0464297007, 0399223001, 0720125001, 0610776001, 0673396002, 0706016003)"
5bbe0c2cbbe0646f01d93a7020ceccbc42b311f780ea3c638bb036610180562d,"List(0706016001, 0706016002, 0372860001, 0610776002, 0759871002, 0372860002, 0464297007, 0399223001, 0720125001, 0610776001, 0673396002, 0706016003)"
f2f2d11004b7af5a324d2e8c8d848a95b28df611b7d092601e0214df46d159cc,"List(0706016001, 0706016002, 0372860001, 0610776002, 0759871002, 0372860002, 0464297007, 0399223001, 0720125001, 0610776001, 0673396002, 0706016003)"
f5cff208cf75282709a28aa4ecfa622202003da10d9c7b8bd13be867c89928a6,"List(0706016001, 0706016002, 0372860001, 0610776002, 0759871002, 0372860002, 0464297007, 0399223001, 0720125001, 0610776001, 0673396002, 0706016003)"


In [0]:
# ========================================
# EVALUATION METRICS
# ========================================
# MAP@K = Mean Average Precision at K (how well are relevant items ranked?)
# Recall@K = What fraction of actual purchases did we capture?

def evaluate_recommendations(predictions_df, ground_truth_df, k=12):
    """
    Calculate MAP@K and Recall@K for recommendations
    """
    # Join predictions with ground truth
    eval_df = predictions_df.join(ground_truth_df, on="customer_id", how="inner")
    
    # Convert to Pandas for metric calculation
    eval_pd = eval_df.toPandas()
    
    def average_precision_at_k(actual, predicted, k):
        """Calculate AP@K for a single customer"""
        # Convert to lists to handle numpy arrays
        actual = list(actual) if actual is not None else []
        predicted = list(predicted) if predicted is not None else []
        
        if len(actual) == 0:
            return 0.0
        
        predicted = predicted[:k]
        score = 0.0
        num_hits = 0.0
        
        for i, p in enumerate(predicted):
            if p in actual and p not in predicted[:i]:
                num_hits += 1.0
                score += num_hits / (i + 1.0)
        
        return score / min(len(actual), k)
    
    def recall_at_k(actual, predicted, k):
        """Calculate Recall@K for a single customer"""
        # Convert to lists to handle numpy arrays
        actual = list(actual) if actual is not None else []
        predicted = list(predicted) if predicted is not None else []
        
        if len(actual) == 0:
            return 0.0
        predicted = set(predicted[:k])
        actual = set(actual)
        return len(predicted & actual) / len(actual)
    
    # Calculate metrics for each customer
    eval_pd['ap'] = eval_pd.apply(
        lambda row: average_precision_at_k(row['actual_articles'], row['predicted_articles'], k), 
        axis=1
    )
    eval_pd['recall'] = eval_pd.apply(
        lambda row: recall_at_k(row['actual_articles'], row['predicted_articles'], k), 
        axis=1
    )
    
    # Calculate mean metrics
    map_score = eval_pd['ap'].mean()
    recall_score = eval_pd['recall'].mean()
    
    return {
        'MAP@K': map_score,
        'Recall@K': recall_score,
        'K': k,
        'num_customers': len(eval_pd)
    }

# Run evaluation
metrics = evaluate_recommendations(predictions_df, test_ground_truth, k=K)

print("=" * 60)
print("📊 BASELINE MODEL EVALUATION RESULTS")
print("=" * 60)
print(f"   Model:          Popularity Baseline")
print(f"   Strategy:       Recommend top-{K} popular items to everyone")
print(f"   Test Customers: {metrics['num_customers']:,}")
print()
print(f"   📈 MAP@{K}:      {metrics['MAP@K']:.6f}")
print(f"   📈 Recall@{K}:   {metrics['Recall@K']:.6f}")
print("=" * 60)
print("\n💡 These are the numbers we need to BEAT with our real model!")

📊 BASELINE MODEL EVALUATION RESULTS
   Model:          Popularity Baseline
   Strategy:       Recommend top-12 popular items to everyone
   Test Customers: 75,481

   📈 MAP@12:      0.002989
   📈 Recall@12:   0.008016

💡 These are the numbers we need to BEAT with our real model!


In [0]:
# ========================================
# MLFLOW EXPERIMENT TRACKING
# ========================================
# Log our baseline model so we can compare with future models

# Set experiment name (using your combined ID)
experiment_name = "/Shared/hm_recommendation_60304248"

# Create experiment if it doesn't exist
mlflow.set_experiment(experiment_name)

# Log the baseline run
with mlflow.start_run(run_name="baseline_popularity") as run:
    
    # Log parameters
    mlflow.log_param("model_type", "popularity_baseline")
    mlflow.log_param("k", K)
    mlflow.log_param("train_start_date", "2018-09-20")
    mlflow.log_param("train_end_date", str(split_date))
    mlflow.log_param("test_start_date", str(split_date))
    mlflow.log_param("test_end_date", "2020-09-22")
    mlflow.log_param("train_transactions", train_count)
    mlflow.log_param("test_customers", metrics['num_customers'])
    
    # Log metrics
    mlflow.log_metric("map_at_12", metrics['MAP@K'])
    mlflow.log_metric("recall_at_12", metrics['Recall@K'])
    
    # Log top-K articles as artifact
    top_k_pandas.to_csv("/tmp/baseline_top_12_articles.csv", index=False)
    mlflow.log_artifact("/tmp/baseline_top_12_articles.csv")
    
    # Get run ID
    baseline_run_id = run.info.run_id
    
print("✅ MLflow experiment logged!")
print(f"   Experiment: {experiment_name}")
print(f"   Run ID: {baseline_run_id}")
print(f"\n📊 View in MLflow UI: Click 'Experiments' in left sidebar")

✅ MLflow experiment logged!
   Experiment: /Shared/hm_recommendation_60304248
   Run ID: d9e6f4d13a504e99ab372d1dabc292e9

📊 View in MLflow UI: Click 'Experiments' in left sidebar


In [0]:
# ========================================
# SAVE BASELINE RESULTS
# ========================================
# Save results for comparison with future models

from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

baseline_results = spark.createDataFrame([{
    "model_name": "popularity_baseline",
    "run_id": baseline_run_id,
    "map_at_12": float(metrics['MAP@K']),
    "recall_at_12": float(metrics['Recall@K']),
    "k": K,
    "train_transactions": train_count,
    "test_customers": metrics['num_customers'],
    "run_date": datetime.now().isoformat()
}])

# Save to curated container
baseline_results.write.mode("overwrite").parquet(f"{MODEL_OUTPUT}baseline_results/")

print("✅ Baseline results saved!")
print(f"   Location: {MODEL_OUTPUT}baseline_results/")
display(baseline_results)

✅ Baseline results saved!
   Location: abfss://curated-p1@ragadziyada.dfs.core.windows.net/models/baseline_results/


k,map_at_12,model_name,recall_at_12,run_date,run_id,test_customers,train_transactions
12,0.0029885495472643524,popularity_baseline,0.008016076330255405,2026-04-10T10:38:18.747687,d9e6f4d13a504e99ab372d1dabc292e9,75481,28571904


# 📊 Notebook Summary: Baseline Model Complete!

## What We Did
1. ✅ Loaded curated data from Phase 1 (28.8M transactions)
2. ✅ Created **temporal train/test split** (no data leakage!)
   - Train: 2018-09-20 to 2020-09-15 (28.5M transactions)
   - Test: 2020-09-15 to 2020-09-22 (241K transactions, 75K customers)
3. ✅ Built **Popularity Baseline** model (recommend top-12 items to everyone)
4. ✅ Evaluated with MAP@12 and Recall@12
5. ✅ Logged experiment to **MLflow**
6. ✅ Saved results to Azure Data Lake

## Baseline Results

| Metric | Score |
|--------|-------|
| MAP@12 | 0.00299 |
| Recall@12 | 0.00802 |

## Next Steps
**Notebook 06:** Train Personalized Popularity model using department preferences
- Learns personalized user-item preferences
- Should significantly outperform this baseline!

---
*Phase 2 - Model Development | H&M Fashion Recommendations | Team 60304248*